In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.platypus.flowables import HRFlowable

# =========================================================
# CARREGA .ENV
# =========================================================

load_dotenv()

# =========================================================
# CONFIGURAÇÃO
# =========================================================

PASTA_SAIDA = "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente"

os.makedirs(PASTA_SAIDA, exist_ok=True)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# =========================================================
# PROMPTS POR CATEGORIA
# =========================================================

prompts = {
    "faqs": """
Você é um especialista em documentação hospitalar.

Gere 30 FAQs médicas hospitalares.

Formato JSON:
[
  {
    "id": 1,
    "tipo": "faq",
    "categoria": "",
    "conteudo": {
      "pergunta": "",
      "resposta": ""
    }
  }
]
""",

    "protocolos": """
Você é um especialista em protocolos hospitalares.

Gere 20 protocolos clínicos sintéticos hospitalares.

Formato JSON:
[
  {
    "id": 1,
    "tipo": "protocolo",
    "categoria": "",
    "conteudo": {
      "titulo": "",
      "descricao": "",
      "conduta": ""
    }
  }
]
""",

    "laudos": """
Você é um especialista em laudos médicos.

Gere 20 laudos médicos sintéticos.

Formato JSON:
[
  {
    "id": 1,
    "tipo": "laudo",
    "categoria": "",
    "conteudo": {
      "exame": "",
      "resultado": "",
      "conclusao": ""
    }
  }
]
""",

    "receitas": """
Você é um especialista em prescrições médicas.

Gere 20 receitas médicas sintéticas.

Formato JSON:
[
  {
    "id": 1,
    "tipo": "receita",
    "categoria": "",
    "conteudo": {
      "medicamento": "",
      "dosagem": "",
      "orientacao": ""
    }
  }
]
""",

    "triagens": """
Você é um especialista em triagem hospitalar.

Gere 20 triagens sintéticas.

Formato JSON:
[
  {
    "id": 1,
    "tipo": "triagem",
    "categoria": "",
    "conteudo": {
      "queixa_principal": "",
      "sinais_vitais": "",
      "classificacao_risco": ""
    }
  }
]
""",

    "evolucoes": """
Você é um especialista em evolução médica hospitalar.

Gere 20 evoluções médicas sintéticas.

Formato JSON:
[
  {
    "id": 1,
    "tipo": "evolucao",
    "categoria": "",
    "conteudo": {
      "quadro_clinico": "",
      "conduta": "",
      "observacao": ""
    }
  }
]
"""
}

# =========================================================
# ESTILO PDF
# =========================================================

styles = getSampleStyleSheet()

# =========================================================
# FUNÇÃO GERAR PDF
# =========================================================

def gerar_pdf(nome_arquivo, dados):

    pdf_path = os.path.join(
        PASTA_SAIDA,
        f"{nome_arquivo}.pdf"
    )

    pdf = SimpleDocTemplate(
        pdf_path,
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )

    elementos = []

    titulo = Paragraph(
        f"<b>{nome_arquivo.upper()}</b>",
        styles["Title"]
    )

    elementos.append(titulo)
    elementos.append(Spacer(1, 20))

    for item in dados:

        texto = f"""
        <b>ID:</b> {item.get("id")}<br/><br/>
        <b>Tipo:</b> {item.get("tipo")}<br/><br/>
        <b>Categoria:</b> {item.get("categoria")}<br/><br/>
        <b>Conteúdo:</b><br/><br/>
        {json.dumps(item.get("conteudo"), ensure_ascii=False, indent=2)}
        """

        elementos.append(
            Paragraph(texto.replace("\n", "<br/>"), styles["BodyText"])
        )

        elementos.append(Spacer(1, 12))
        elementos.append(HRFlowable(width="100%"))
        elementos.append(Spacer(1, 12))

    pdf.build(elementos)

    print(f"PDF salvo: {pdf_path}")

# =========================================================
# LOOP DAS CATEGORIAS
# =========================================================

for nome_categoria, prompt in prompts.items():

    print(f"\nGerando: {nome_categoria}")

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.8
    )

    conteudo = response.choices[0].message.content

    # =====================================================
    # REMOVE ```json
    # =====================================================

    conteudo = conteudo.replace("```json", "")
    conteudo = conteudo.replace("```", "")
    conteudo = conteudo.strip()

    # =====================================================
    # CONVERTE JSON
    # =====================================================

    dados = json.loads(conteudo)

    # =====================================================
    # SALVA JSON
    # =====================================================

    json_path = os.path.join(
        PASTA_SAIDA,
        f"{nome_categoria}.json"
    )

    with open(json_path, "w", encoding="utf-8") as arquivo:
        json.dump(
            dados,
            arquivo,
            ensure_ascii=False,
            indent=2
        )

    print(f"JSON salvo: {json_path}")

    # =====================================================
    # GERA PDF
    # =====================================================

    gerar_pdf(nome_categoria, dados)

print("\nFINALIZADO.")


Gerando: faqs
JSON salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/faqs.json
PDF salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/faqs.pdf

Gerando: protocolos
JSON salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/protocolos.json
PDF salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/protocolos.pdf

Gerando: laudos
JSON salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/laudos.json
PDF salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/laudos.pdf

Gerando: receitas
JSON salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/receitas.json
PDF salvo: /content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/arquivos-assistente/receitas.pdf

Gerando: triagens
JSON salvo: /content/drive/MyDrive/Te